# Remote server smoke tests

This notebook queries the running MaterialX remote server endpoints and prints their JSON responses.
Adjust `BASE_URL` if your server runs on a different host/port.

In [ ]:
import json
import requests
from pprint import pprint

BASE_URL = 'http://localhost:2907'

def show_response(r):
    try:
        pprint(r.json())
    except Exception:
        print(r.status_code, r.text)

In [ ]:
# Health check
r = requests.get(f'{BASE_URL}/health')
print('GET /health ->', r.status_code)
show_response(r)

In [ ]:
# List materials
r = requests.get(f'{BASE_URL}/materials')
print('GET /materials ->', r.status_code)
show_response(r)
materials = r.json() if r.status_code == 200 else []
sample_name = materials[0]['name'] if materials else None
print('sample_name =', sample_name)

In [ ]:
# Select material (if any)
if sample_name:
    r = requests.post(f'{BASE_URL}/materials/select', json={'name': sample_name})
    print('POST /materials/select ->', r.status_code)
    show_response(r)
else:
    print('No materials to select')

In [ ]:
# Get shader for currently selected material
r = requests.get(f'{BASE_URL}/shader')
print('GET /shader ->', r.status_code)
show_response(r)

In [ ]:
# Geometry list and selection
r = requests.get(f'{BASE_URL}/geometry')
print('GET /geometry ->', r.status_code)
show_response(r)
geom_list = r.json().get('geometry', []) if r.status_code==200 else []
if geom_list:
    gid = geom_list[0]
    r2 = requests.post(f'{BASE_URL}/geometry', json={'id': gid})
    print('POST /geometry ->', r2.status_code)
    show_response(r2)
else:
    print('No geometry available')

In [ ]:
# Lights get/set
r = requests.get(f'{BASE_URL}/lights')
print('GET /lights ->', r.status_code)
show_response(r)
payload = {'envLightIntensity': 1.5, 'lightRotation': 30.0}
r2 = requests.post(f'{BASE_URL}/lights', json=payload)
print('POST /lights ->', r2.status_code)
show_response(r2)

In [ ]:
# Camera get/set
r = requests.get(f'{BASE_URL}/camera')
print('GET /camera ->', r.status_code)
show_response(r)
payload = {'position':[0,0,5], 'target':[0,0,0], 'viewAngle':30, 'zoom':1.0}
print('kek')
r2 = requests.post(f'{BASE_URL}/camera', json=payload)
print('kek2')
print('POST /camera ->', r2.status_code)
show_response(r2)

In [ ]:
# Test: fetch vertex stage, add comment, POST only vertex back, then verify
r = requests.get(f'{BASE_URL}/shader')
print('GET /shader ->', r.status_code)
data = r.json() if r.status_code==200 else {}
stages = data.get('stages', {})
vertex = stages.get('vertex', '')
fragment = stages.get('fragment', '')
if not vertex:
    print('No vertex stage available to test')
else:
    # Prepend a simple comment to the vertex shader source to test override
    commented_vertex = '/* override-test: added comment */' + vertex
    payload = {'vertex': commented_vertex}
    r2 = requests.post(f'{BASE_URL}/shader', json=payload)
    print('POST /shader ->', r2.status_code)
    try:
        pprint(r2.json())
    except Exception:
        print(r2.status_code, r2.text)
    # Verify override is returned by GET /shader
    r3 = requests.get(f'{BASE_URL}/shader')
    
    print('GET /shader (verify) ->', r3.status_code)
    try:
        resp = r3.json(); pprint(resp)
        v2 = resp.get('stages', {}).get('vertex', '')
        if v2.startswith('/* override-test: added comment */'):
            print('Vertex override verified')
        else:
            print('Vertex override not present')
    except Exception:
        print('Verification failed', r3.status_code, r3.text)

In [ ]:
# Test: fetch fragment stage, add comment, POST only fragment back, then verify
r = requests.get(f'{BASE_URL}/shader')
print('GET /shader ->', r.status_code)
data = r.json() if r.status_code==200 else {}
stages = data.get('stages', {})
vertex = stages.get('vertex', '')
fragment = stages.get('fragment', '')
if not fragment:
    print('No fragment stage available to test')
else:
    # Prepend a simple comment to the fragment shader source to test override
    commented_fragment = '/* override-test: fragment comment */' + fragment
    payload = {'fragment': commented_fragment}
    r2 = requests.post(f'{BASE_URL}/shader', json=payload)
    print('POST /shader ->', r2.status_code)
    try:
        pprint(r2.json())
    except Exception:
        print(r2.status_code, r2.text)
    # Verify override is returned by GET /shader
    r3 = requests.get(f'{BASE_URL}/shader')
    print('GET /shader (verify) ->', r3.status_code)
    try:
        resp = r3.json(); pprint(resp)
        f2 = resp.get('stages', {}).get('fragment', '')
        if f2.startswith('/* override-test: fragment comment */'):
            print('Fragment override verified')
        else:
            print('Fragment override not present')
    except Exception:
        print('Verification failed', r3.status_code, r3.text)

In [ ]:
# Get shader for currently selected material
r = requests.get(f'{BASE_URL}/shader')
print('GET /shader ->', r.status_code)
show_response(r)

In [ ]:
# Test: Post a simple fragment shader that forces red output and verify override
# This shader writes red to the first output (out1)
red_fragment = ('/* override-test: force red */\n'
                '#version 400\n'
                '\n'
                'out vec4 out1;\n'
                'void main() { out1 = vec4(1.0, 0.0, 0.0, 1.0); }')
payload = {'fragment': red_fragment}
r = requests.post(f'{BASE_URL}/shader', json=payload)
print('POST /shader (force red) ->', r.status_code)
show_response(r)
# Verify GET returns the override (merged), and check fragment stage starts with our marker
r2 = requests.get(f'{BASE_URL}/shader')
print('GET /shader (verify red) ->', r2.status_code)
try:
    data = r2.json() if r2.status_code==200 else {}
    frag = data.get('stages', {}).get('fragment', '')
    if frag.startswith('/* override-test: force red */'):
        print('Red fragment override present')
    else:
        print('Red fragment override NOT present')
except Exception:
    print('Failed to verify GET /shader', r2.status_code, r2.text)